# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


I chose Lane 2 Refresh / Content Opportunity Scoring.
ML task type: Ranking / Scoring
The goal is to rank pages based on how likely they are to need a content refresh, expansion, or monitoring. Instead of predicting Google's ranking algorithm, the model helps SEO teams prioritize which pages should be reviewed first. This is a ranking problem because the output is an ordered list of pages rather than a simple yes/no prediction.

**The action it supports:** each week an SEO editor takes the top of the list and decides, page by page, whether to refresh it, expand it, or just keep monitoring it. The output is decision support for that editor, not an automatic change to any page.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label`, which is 1 when a page's `trend_direction` is "down" and 0 otherwise.

There is no column that directly says a page "needs a refresh", so I use observed decline as the proxy. A page whose search traffic is going down is the one an editor most likely needs to look at. This label comes from what actually happened to the page's traffic, not from a rule I made up, so the model learns something about the real world instead of copying my own rule back to me.

Metrics like CTR, average position, impressions and engagement rate are **inputs** (features), not the target.

**Leakage note:** because the label is built from `trend_direction`, which comes from `trend_pct`, neither of those two columns can ever be used as a feature.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will evaluate the ranking using **Precision@50**.

Precision@50 measures how many of the top 50 pages recommended by the model are actually good candidates for refresh or review. A higher Precision@50 means the SEO team spends more time reviewing pages that are likely to benefit from action rather than pages that do not need attention.

**Cost of a wrong call:** a false positive in the top 50 wastes an editor's time on a page that was fine. A missed declining page keeps losing traffic until someone notices. Editor time is the limited resource, so precision at the top of the list matters more than overall accuracy.

**What counts as good:** the base rate of declining pages is about 54%, so picking 50 pages at random would give Precision@50 of about 0.54. The model has to clearly beat both that and a simple fixed rule.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print("Unique content_id:", df["content_id"].nunique())
df.head()

Rows: 30000
Columns: 44
Unique content_id: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Each row represents **one content page** with aggregated search and engagement metrics.
The model will make one prediction for each page, producing a ranked list of pages that should be reviewed first.

In [2]:
# Sketch of the target column (same definition as scripts/01_prepare_features.py)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["is_declining_label"].value_counts())
print("Base rate:", round(df["is_declining_label"].mean(), 3))
df[["content_id", "content_type", "impressions_90d", "ctr", "avg_position", "is_declining_label"]].head(10)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate: 0.542


,content_id,content_type,impressions_90d,ctr,avg_position,is_declining_label
0,content_304f48230142,keyword article,3803,0.76,10.6,1
1,content_a1fb4e703a9e,keyword article,15320,0.05,20.3,1
2,content_9aa793d4d895,keyword article,12581,0.09,36.5,1
3,content_331d6c4de07b,keyword article,11751,0.49,6.2,0
4,content_d99b7a2d90ca,keyword article,19140,0.13,44.0,1
5,content_d4084a4bc775,keyword article,3970,0.03,8.5,1
6,content_9a34b442b552,keyword article,20,0.00,7.0,1
7,content_a63219c6e95a,keyword article,1724,0.06,21.2,0
8,content_5e6c160719bc,keyword article,32574,0.09,46.0,1
9,content_c27558df2b0c,keyword article,1240,0.16,4.9,1


The target column is already fairly balanced (54.2% declining), so the model is not chasing a rare event. The hard part is putting the *right* declining pages at the top.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as "refresh every page with low CTR" is too simplistic because page performance depends on several factors at the same time, including impressions, average position, engagement rate, search volume, competition, and content type.

Low CTR on page 30 means something very different from low CTR on page 3, and a page with only a handful of impressions can look bad on every rate just from noise. The signals interact, and writing every combination by hand as if-statements doesn't scale.

I tested this in my pipeline. The hand-written rules baseline got **Precision@50 = 0.24**, which is worse than picking at random (about 0.54). The Random Forest got **0.68** when validated on clients it never saw during training. That is directional evidence from anonymized starter data, not a guarantee, but it shows that a fixed rule leaves real value on the table.

Machine learning can learn patterns across multiple features simultaneously and produce a better ranking than a single if-statement while still supporting human decision-making.

In [3]:
print("Pages:", len(df))
print("Average CTR (%):", round(df["ctr"].mean(), 4))
print("Average Position:", round(df[df["avg_position"] > 0]["avg_position"].mean(), 2))
print()
print("Content Types")
print(df["content_type"].value_counts())

Pages: 30000
Average CTR (%): 0.5107
Average Position: 17.03

Content Types
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.